# YOLOv8-cls Training — Skin Lesion Classification (Split_smol / Roboflow)

This notebook trains a YOLOv8 classification model on your 9-class skin lesion dataset,
handles class imbalance via oversampling, and reports per-class precision/recall/F1 —
not just overall accuracy, which can be misleading with imbalanced medical image data.

**Before running:** Runtime > Change runtime type > GPU (T4 or better).

## 1. Setup

In [1]:
!pip install -q ultralytics

import os, shutil, random
from pathlib import Path
from collections import Counter
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


## 2. Get your data — via Roboflow

⚠️ **Never share your real API key publicly** (GitHub, forums, shared notebooks). Prefer
Colab's **Secrets** manager (key icon in the left sidebar): add a secret named
`ROBOFLOW_API_KEY`, then read it with `from google.colab import userdata` instead of
pasting the key directly below.

In [2]:
!pip install -q roboflow

from roboflow import Roboflow

# Option A (quick, less safe): paste your key directly — fine for a private/local notebook only
rf = Roboflow(api_key="LowmTu68QhcYNV87xaN7")

# Option B (safer): use Colab Secrets instead of the line above
# from google.colab import userdata
# rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))

project = rf.workspace("moman-butt").project("skin_disease-ub2iy")
version = project.version(1)
dataset = version.download("folder")

DATA_ROOT = dataset.location
# Roboflow's classification export produces:
#   <DATA_ROOT>/train/<class_name>/*.jpg
#   <DATA_ROOT>/valid/<class_name>/*.jpg   <-- note: "valid", not "val"
print("Dataset downloaded to:", DATA_ROOT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.6 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Skin_Disease-1 in folder:: 100%|██████████| 847/847 [00:00<00:00, 7634.91it/s]

Dataset downloaded to: /content/Skin_Disease-1


## 3. Audit the dataset BEFORE training

Check class imbalance, and flag possible train/valid leakage for the
Tinea Ringworm class (its filenames show signs of pre-existing augmentation,
e.g. `aug_0_...`, which can leak near-duplicate images across the split).

In [3]:
def audit(root):
    for split in ["train", "valid"]:
        print(f"\n--- {split} ---")
        split_path = Path(root) / split
        counts = {}
        for cls_dir in sorted(split_path.iterdir()):
            if cls_dir.is_dir():
                counts[cls_dir.name] = len(list(cls_dir.glob("*")))
        for k, v in counts.items():
            print(f"{k:35s}: {v}")
        print(f"TOTAL: {sum(counts.values())}, "
              f"min class: {min(counts.values())}, max class: {max(counts.values())}, "
              f"imbalance ratio: {max(counts.values())/min(counts.values()):.1f}x")

audit(DATA_ROOT)


--- train ---
actinic keratosis                  : 80
atopic dermatitis                  : 65
benign keratosis                   : 80
dermatofibroma                     : 80
melanocytic nevus                  : 80
melanoma                           : 80
squamous cell carcinoma            : 80
tinea ringworm candidiasis         : 44
vascular lesion                    : 80
TOTAL: 669, min class: 44, max class: 80, imbalance ratio: 1.8x

--- valid ---
actinic keratosis                  : 20
atopic dermatitis                  : 16
benign keratosis                   : 20
dermatofibroma                     : 20
melanocytic nevus                  : 20
melanoma                           : 20
squamous cell carcinoma            : 20
tinea ringworm candidiasis         : 20
vascular lesion                    : 20
TOTAL: 176, min class: 16, max class: 20, imbalance ratio: 1.2x


In [4]:
# Quick leakage check for augmented filenames (aug_0_, aug_1_, etc.)
# If the same base filename appears in both train and valid for a class, flag it.
def check_leakage(root, class_name):
    def base_names(split):
        p = Path(root) / split / class_name
        return set(f.stem.split("aug_")[-1].split("_", 1)[-1] if "aug_" in f.stem else f.stem
                   for f in p.glob("*"))
    tr, va = base_names("train"), base_names("valid")
    overlap = tr & va
    print(f"{class_name}: {len(overlap)} potentially overlapping base names between train/valid")
    return overlap

check_leakage(DATA_ROOT, "Tinea Ringworm Candidiasis")

Tinea Ringworm Candidiasis: 0 potentially overlapping base names between train/valid


set()

## 4. Handle class imbalance — with REAL augmentation, not plain duplication

YOLOv8-cls doesn't support per-class loss weights directly, so we oversample minority
classes in a **copy** of the train folder. Earlier this duplicated files exactly —
which lets the model partly memorize repeats instead of learning general features.
This version applies a **different random transform to each duplicate** (via
`albumentations`) so oversampled images are genuinely new training signal, not clones.
Validation data is never touched.

In [5]:
!pip install -q albumentations

import albumentations as A
import cv2

# Mild transforms only — enough to add variety without changing diagnostic features
# (lesion color/shape). Avoid heavy color shifts for a medical image classifier.
augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=25, border_mode=cv2.BORDER_REFLECT, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), p=0.5),
])

def build_balanced_train(src_root, dst_root, target_per_class=None):
    src_train = Path(src_root) / "train"
    dst_train = Path(dst_root) / "train"
    if dst_train.exists():
        shutil.rmtree(dst_train)
    dst_train.mkdir(parents=True)

    class_files = {d.name: list(d.glob("*")) for d in src_train.iterdir() if d.is_dir()}
    if target_per_class is None:
        target_per_class = max(len(v) for v in class_files.values())  # match the majority class

    for cls, files in class_files.items():
        out_dir = dst_train / cls
        out_dir.mkdir(parents=True)
        # copy originals untouched
        for f in files:
            shutil.copy(f, out_dir / f.name)

        # oversample minority classes with genuinely augmented copies
        n_needed = target_per_class - len(files)
        i = 0
        while n_needed > 0:
            f = random.choice(files)
            img = cv2.imread(str(f))
            if img is None:
                n_needed -= 1
                continue
            augmented = augmentor(image=img)["image"]
            new_name = f"aug{i}_{f.stem}.jpg"
            cv2.imwrite(str(out_dir / new_name), augmented)
            n_needed -= 1
            i += 1

    # valid stays untouched — never balance/augment validation data
    dst_val = Path(dst_root) / "valid"
    if dst_val.exists():
        shutil.rmtree(dst_val)
    shutil.copytree(Path(src_root) / "valid", dst_val)

BALANCED_ROOT = "/content/balanced_split_smol"
build_balanced_train(DATA_ROOT, BALANCED_ROOT)
audit(BALANCED_ROOT)


--- train ---
actinic keratosis                  : 80
atopic dermatitis                  : 80
benign keratosis                   : 80
dermatofibroma                     : 80
melanocytic nevus                  : 80
melanoma                           : 80
squamous cell carcinoma            : 80
tinea ringworm candidiasis         : 80
vascular lesion                    : 80
TOTAL: 720, min class: 80, max class: 80, imbalance ratio: 1.0x

--- valid ---
actinic keratosis                  : 20
atopic dermatitis                  : 16
benign keratosis                   : 20
dermatofibroma                     : 20
melanocytic nevus                  : 20
melanoma                           : 20
squamous cell carcinoma            : 20
tinea ringworm candidiasis         : 20
vascular lesion                    : 20
TOTAL: 176, min class: 16, max class: 20, imbalance ratio: 1.2x


## 5. Train — v3: rolled back over-regularization, kept real augmentation

**Results so far:**
- v1 (baseline): 81.8% top1, melanoma recall 40%
- v2 (dropout 0.35, weight_decay 0.001, imgsz 256): 80.7% top1, melanoma recall **still 40%**,
  best result found at epoch 20 despite patience=35 — the stronger regularization didn't help
  and slightly hurt.

**What this run does differently:**
- Reverts `dropout` and `weight_decay` back to v1's values (0.2 / 0.0005) — the v2 increase
  made things marginally worse, not better
- Keeps the real-augmentation oversampling from v2 (that part is a genuine improvement,
  unrelated to the regularization regression)
- Keeps `imgsz=256` since it's not clearly harmful, just adjust down to 224 if this run
  also plateaus at the same accuracy (isolates whether resolution matters at all)

**Important reframe:** two full runs now show melanoma recall stuck at 40% regardless of
these hyperparameters. That strongly suggests the ceiling here isn't tuning — it's that the
model hasn't seen enough varied real melanoma/nevus examples to separate them. Cell 6b below
adds a targeted fine-tune stage specifically for this confusable pair, which is likely to
matter more than any further hyperparameter change.

In [6]:
model = YOLO("yolov8m-cls.pt")

results = model.train(
    data=BALANCED_ROOT,
    epochs=100,
    patience=25,
    imgsz=256,
    batch=32,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    dropout=0.2,             # reverted from 0.35 (v2) — that was worse, not better
    weight_decay=0.0005,     # reverted from 0.001 (v2)
    mixup=0.1,
    auto_augment="randaugment",
    erasing=0.4,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    degrees=15, flipud=0.5, fliplr=0.5,
    val=True,
    plots=True,
    project="skin_lesion_runs",
    name="yolov8m_cls_v3",
    seed=42,
)

Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/balanced_split_smol, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_cls_v3, nbs=64, nms=F

## 6. Evaluate — with MANUAL Test-Time Augmentation

⚠️ **The previous `augment=True` flag doesn't actually work for classification models** —
Ultralytics printed `ClassificationModel does not support 'augment=True' prediction` and
silently ran a normal single-pass prediction instead. This version implements TTA manually:
predicting on the original image plus a horizontal flip and averaging the two probability
vectors. This is a legitimate small accuracy boost (previously not actually applied).

In [7]:
metrics = model.val(data=BALANCED_ROOT, split="val")  # native augment=True is a no-op for cls models, removed
print("Top1:", metrics.top1, "| Top5:", metrics.top5)

Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8m-cls summary (fused): 42 layers, 15,774,185 parameters, 0 gradients, 41.6 GFLOPs
train: /content/balanced_split_smol/train... found 720 images in 9 classes ✅ 
val: /content/balanced_split_smol/valid... found 176 images in 9 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 250.3±77.2 MB/s, size: 7.3 KB)
val: Scanning /content/balanced_split_smol/valid... 176 images, 0 corrupt: 100% ━━━━━━━━━━━━ 176/176 52.7Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 11/11 8.5it/s 1.3s
                   all      0.852      0.994
Speed: 0.6ms preprocess, 4.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val
Top1: 0.8522727489471436 | Top5: 0.9943181872367859


In [8]:
import numpy as np
import cv2
from sklearn.metrics import classification_report, confusion_matrix

val_dir = Path(BALANCED_ROOT) / "valid"
class_names = sorted([d.name for d in val_dir.iterdir() if d.is_dir()])

def predict_with_tta(model, img_path):
    img = cv2.imread(str(img_path))
    img_flipped = cv2.flip(img, 1)  # horizontal flip
    p1 = model.predict(img, verbose=False)[0].probs.data
    p2 = model.predict(img_flipped, verbose=False)[0].probs.data
    avg = (p1 + p2) / 2
    return class_names[int(avg.argmax())]

y_true, y_pred = [], []
for cls in class_names:
    for f in (val_dir / cls).glob("*"):
        pred_cls = predict_with_tta(model, f)
        y_true.append(cls)
        y_pred.append(pred_cls)

print(classification_report(y_true, y_pred, digits=3))
print(confusion_matrix(y_true, y_pred, labels=class_names))

                            precision    recall  f1-score   support

         actinic keratosis      0.667     0.800     0.727        20
         atopic dermatitis      0.842     1.000     0.914        16
          benign keratosis      1.000     0.950     0.974        20
            dermatofibroma      0.947     0.900     0.923        20
         melanocytic nevus      0.773     0.850     0.810        20
                  melanoma      0.750     0.600     0.667        20
   squamous cell carcinoma      0.700     0.700     0.700        20
tinea ringworm candidiasis      1.000     0.850     0.919        20
           vascular lesion      1.000     1.000     1.000        20

                  accuracy                          0.847       176
                 macro avg      0.853     0.850     0.848       176
              weighted avg      0.853     0.847     0.847       176

[[16  0  0  1  0  0  3  0  0]
 [ 0 16  0  0  0  0  0  0  0]
 [ 0  0 19  0  0  1  0  0  0]
 [ 1  0  0 18  0  0  1 

## 6c. Targeted fine-tune: Melanoma vs. Melanocytic Nevus

Two runs show melanoma recall stuck at 40%, mostly confused with nevus. This does a short
second training pass using **only these two classes**, with a low learning rate, to sharpen
specifically that boundary without disturbing the other 7 classes already performing well.

This freezes everything except the classifier head — a light touch that adjusts the decision
boundary for these two classes without forgetting the rest.

In [9]:
BEST_PT = "/content/runs/classify/skin_lesion_runs/yolov8m_cls_v3/weights/best.pt"

import shutil
from pathlib import Path
from ultralytics import YOLO

FOCUS_ROOT = "/content/melanoma_nevus_focus"
focus_train = Path(FOCUS_ROOT) / "train"
focus_valid = Path(FOCUS_ROOT) / "valid"
for p in [focus_train, focus_valid]:
    if p.exists():
        shutil.rmtree(p)
    p.mkdir(parents=True)

for cls in ["melanoma", "melanocytic nevus"]:
    shutil.copytree(Path(BALANCED_ROOT) / "train" / cls, focus_train / cls)
    shutil.copytree(Path(BALANCED_ROOT) / "valid" / cls, focus_valid / cls)

finetune_model = YOLO(BEST_PT)

finetune_results = finetune_model.train(
    data=FOCUS_ROOT,
    epochs=30,
    patience=15,
    imgsz=256,
    batch=16,
    lr0=0.0001,
    freeze=9,     # fixed: was 10 (froze everything, including the head). 9 leaves the classifier head trainable.
    optimizer="AdamW",
    val=True,
    plots=True,
    project="skin_lesion_runs",
    name="melanoma_nevus_finetune",
    seed=42,
)

FINETUNE_BEST_PT = str(finetune_results.save_dir / "weights" / "best.pt")
print("Fine-tuned checkpoint saved at:", FINETUNE_BEST_PT)

finetune_metrics = finetune_model.val(data=FOCUS_ROOT, split="val")
print("Focused Top1:", finetune_metrics.top1)

Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/melanoma_nevus_focus, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=9, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/classify/skin_lesion_runs/yolov8m_cls_v3/weights/best.pt, momentum=0.937, mosaic=

In [10]:
from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix

main_model = YOLO(BEST_PT)
specialist_model = YOLO(FINETUNE_BEST_PT)

val_dir = Path(BALANCED_ROOT) / "valid"
class_names = sorted([d.name for d in val_dir.iterdir() if d.is_dir()])
specialist_classes = sorted(["melanoma", "melanocytic nevus"])  # matches specialist's own class order

CONFUSABLE = {"melanoma", "melanocytic nevus"}

y_true, y_pred = [], []
for cls in class_names:
    for f in (val_dir / cls).glob("*"):
        main_pred = main_model.predict(str(f), verbose=False)[0]
        main_label = class_names[main_pred.probs.top1]

        if main_label in CONFUSABLE:
            # defer to the specialist for the final call
            spec_pred = specialist_model.predict(str(f), verbose=False)[0]
            final_label = specialist_classes[spec_pred.probs.top1]
        else:
            final_label = main_label

        y_true.append(cls)
        y_pred.append(final_label)

print(classification_report(y_true, y_pred, digits=3))
print(confusion_matrix(y_true, y_pred, labels=class_names))

                            precision    recall  f1-score   support

         actinic keratosis      0.680     0.850     0.756        20
         atopic dermatitis      0.842     1.000     0.914        16
          benign keratosis      1.000     0.950     0.974        20
            dermatofibroma      0.950     0.950     0.950        20
         melanocytic nevus      0.611     0.550     0.579        20
                  melanoma      0.526     0.500     0.513        20
   squamous cell carcinoma      0.737     0.700     0.718        20
tinea ringworm candidiasis      1.000     0.850     0.919        20
           vascular lesion      1.000     1.000     1.000        20

                  accuracy                          0.812       176
                 macro avg      0.816     0.817     0.814       176
              weighted avg      0.816     0.812     0.811       176

[[17  0  0  1  0  0  2  0  0]
 [ 0 16  0  0  0  0  0  0  0]
 [ 1  0 19  0  0  0  0  0  0]
 [ 0  0  0 19  0  0  1 

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
# This is the main model from THIS run — the one before the fine-tune step, your best result
shutil.copy(BEST_PT, "/content/drive/MyDrive/skin_lesion_FINAL_best.pt")
print("Saved final model to Google Drive: skin_lesion_FINAL_best.pt")
print("Accuracy: 84.7% | Melanoma recall: 60%")

Mounted at /content/drive
Saved final model to Google Drive: skin_lesion_FINAL_best.pt
Accuracy: 84.7% | Melanoma recall: 60%


In [12]:
!pip install -q isic-cli

!isic image download --search 'diagnosis_3:"Melanoma"' --limit 150 /content/extra_data/melanoma
!isic image download --search 'diagnosis_3:"Nevus"' --limit 150 /content/extra_data/melanocytic_nevus
!isic image download --search 'diagnosis_3:"Solar or actinic keratosis"' --limit 150 /content/extra_data/actinic_keratosis
!isic image download --search 'diagnosis_3:"Squamous cell carcinoma, NOS"' --limit 150 /content/extra_data/squamous_cell_carcinoma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.9 MB/s eta 0:00:00
If you have been granted special permissions, logging in with `isic user login` might return more data.


Successfully downloaded 0 images to /content/extra_data/melanoma/.
Successfully wrote 0 metadata records to /content/extra_data/melanoma/metadata.csv.
Successfully wrote attributions to /content/extra_data/melanoma/attribution.txt.
Successfully wrote 0 license(s) to /content/extra_data/melanoma/licenses.
If you have been granted special permissions, logging in with `isic user login` might return more data.


Successfully downloaded 150 images to /content/extra_data/melanocytic_nevus/.
Successfully wrote 150 metadata records to /content/extra_data/melanocytic_nevus/metadata.csv.
Successfully wrote attributions to /content/extra_data/melanocytic_nevus/attribution.txt.
Successfully wrote 1 license(s) to /content/extra_data/melanocytic_nevus/licenses.
If you have been granted special permissions, logging in w

In [13]:
!isic image download --search 'diagnosis_2:"Malignant melanocytic proliferations (Melanoma)"' --limit 150 /content/extra_data/melanoma

If you have been granted special permissions, logging in with `isic user login` might return more data.


Successfully downloaded 150 images to /content/extra_data/melanoma/.
Successfully wrote 150 metadata records to /content/extra_data/melanoma/metadata.csv.
Successfully wrote attributions to /content/extra_data/melanoma/attribution.txt.
Successfully wrote 1 license(s) to /content/extra_data/melanoma/licenses.


In [14]:
import os
for name in ["melanoma", "melanocytic_nevus", "actinic_keratosis", "squamous_cell_carcinoma"]:
    path = f"/content/extra_data/{name}"
    count = len([f for f in os.listdir(path) if f.endswith('.jpg') or f.endswith('.png')]) if os.path.exists(path) else 0
    print(f"{name}: {count} image files")

melanoma: 150 image files
melanocytic_nevus: 150 image files
actinic_keratosis: 150 image files
squamous_cell_carcinoma: 150 image files


In [15]:
import shutil
from pathlib import Path

extra_map = {
    "melanoma": "/content/extra_data/melanoma",
    "melanocytic nevus": "/content/extra_data/melanocytic_nevus",
    "actinic keratosis": "/content/extra_data/actinic_keratosis",
    "squamous cell carcinoma": "/content/extra_data/squamous_cell_carcinoma",
}

for cls, src in extra_map.items():
    dst = Path(DATA_ROOT) / "train" / cls
    copied = 0
    for f in Path(src).glob("*"):
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            shutil.copy(f, dst / f"isic_extra_{f.name}")
            copied += 1
    print(f"{cls}: copied {copied} new images")

melanoma: copied 150 new images
melanocytic nevus: copied 150 new images
actinic keratosis: copied 150 new images
squamous cell carcinoma: copied 150 new images


In [16]:
def audit(root):
    for split in ["train", "valid"]:
        print(f"\n--- {split} ---")
        split_path = Path(root) / split
        counts = {}
        for cls_dir in sorted(split_path.iterdir()):
            if cls_dir.is_dir():
                counts[cls_dir.name] = len(list(cls_dir.glob("*")))
        for k, v in counts.items():
            print(f"{k:35s}: {v}")
        print(f"TOTAL: {sum(counts.values())}, min: {min(counts.values())}, max: {max(counts.values())}")

audit(DATA_ROOT)


--- train ---
actinic keratosis                  : 230
atopic dermatitis                  : 65
benign keratosis                   : 80
dermatofibroma                     : 80
melanocytic nevus                  : 230
melanoma                           : 230
squamous cell carcinoma            : 230
tinea ringworm candidiasis         : 44
vascular lesion                    : 80
TOTAL: 1269, min: 44, max: 230

--- valid ---
actinic keratosis                  : 20
atopic dermatitis                  : 16
benign keratosis                   : 20
dermatofibroma                     : 20
melanocytic nevus                  : 20
melanoma                           : 20
squamous cell carcinoma            : 20
tinea ringworm candidiasis         : 20
vascular lesion                    : 20
TOTAL: 176, min: 16, max: 20


In [17]:
import albumentations as A
import cv2
import random

augmentor = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=25, border_mode=cv2.BORDER_REFLECT, p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Affine(scale=(0.9, 1.1), translate_percent=(0.0, 0.05), p=0.5),
])

def build_balanced_train(src_root, dst_root, target_per_class=None):
    src_train = Path(src_root) / "train"
    dst_train = Path(dst_root) / "train"
    if dst_train.exists():
        shutil.rmtree(dst_train)
    dst_train.mkdir(parents=True)

    class_files = {d.name: list(d.glob("*")) for d in src_train.iterdir() if d.is_dir()}
    if target_per_class is None:
        target_per_class = max(len(v) for v in class_files.values())

    for cls, files in class_files.items():
        out_dir = dst_train / cls
        out_dir.mkdir(parents=True)
        for f in files:
            shutil.copy(f, out_dir / f.name)

        n_needed = target_per_class - len(files)
        i = 0
        while n_needed > 0:
            f = random.choice(files)
            img = cv2.imread(str(f))
            if img is None:
                n_needed -= 1
                continue
            augmented = augmentor(image=img)["image"]
            cv2.imwrite(str(out_dir / f"aug{i}_{f.stem}.jpg"), augmented)
            n_needed -= 1
            i += 1

    dst_val = Path(dst_root) / "valid"
    if dst_val.exists():
        shutil.rmtree(dst_val)
    shutil.copytree(Path(src_root) / "valid", dst_val)

BALANCED_ROOT = "/content/balanced_split_smol_v2"
build_balanced_train(DATA_ROOT, BALANCED_ROOT)
audit(BALANCED_ROOT)


--- train ---
actinic keratosis                  : 230
atopic dermatitis                  : 230
benign keratosis                   : 230
dermatofibroma                     : 230
melanocytic nevus                  : 230
melanoma                           : 230
squamous cell carcinoma            : 230
tinea ringworm candidiasis         : 230
vascular lesion                    : 230
TOTAL: 2070, min: 230, max: 230

--- valid ---
actinic keratosis                  : 20
atopic dermatitis                  : 16
benign keratosis                   : 20
dermatofibroma                     : 20
melanocytic nevus                  : 20
melanoma                           : 20
squamous cell carcinoma            : 20
tinea ringworm candidiasis         : 20
vascular lesion                    : 20
TOTAL: 176, min: 16, max: 20


In [18]:
from ultralytics import YOLO

model = YOLO("yolov8m-cls.pt")

results = model.train(
    data=BALANCED_ROOT,
    epochs=120,
    patience=30,
    imgsz=256,
    batch=32,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    dropout=0.2,
    weight_decay=0.0005,
    mixup=0.1,
    auto_augment="randaugment",
    erasing=0.4,
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.3,
    degrees=15, flipud=0.5, fliplr=0.5,
    val=True,
    plots=True,
    project="skin_lesion_runs",
    name="yolov8m_cls_v4_moredata",
    seed=42,
)

BEST_PT = str(results.save_dir / "weights" / "best.pt")
print("Best checkpoint:", BEST_PT)

Ultralytics 8.4.132 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/balanced_split_smol_v2, degrees=15, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.3, imgsz=256, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_cls_v4_moredata, n

In [19]:
from sklearn.metrics import classification_report, confusion_matrix

val_dir = Path(BALANCED_ROOT) / "valid"
class_names = sorted([d.name for d in val_dir.iterdir() if d.is_dir()])

y_true, y_pred = [], []
for cls in class_names:
    for f in (val_dir / cls).glob("*"):
        pred = model.predict(str(f), verbose=False)[0]
        y_true.append(cls)
        y_pred.append(class_names[pred.probs.top1])

print(classification_report(y_true, y_pred, digits=3))
print(confusion_matrix(y_true, y_pred, labels=class_names))

                            precision    recall  f1-score   support

         actinic keratosis      0.750     0.750     0.750        20
         atopic dermatitis      0.889     1.000     0.941        16
          benign keratosis      1.000     0.950     0.974        20
            dermatofibroma      0.944     0.850     0.895        20
         melanocytic nevus      0.870     1.000     0.930        20
                  melanoma      0.824     0.700     0.757        20
   squamous cell carcinoma      0.652     0.750     0.698        20
tinea ringworm candidiasis      1.000     0.900     0.947        20
           vascular lesion      1.000     1.000     1.000        20

                  accuracy                          0.875       176
                 macro avg      0.881     0.878     0.877       176
              weighted avg      0.881     0.875     0.875       176

[[15  0  0  1  0  1  3  0  0]
 [ 0 16  0  0  0  0  0  0  0]
 [ 1  0 19  0  0  0  0  0  0]
 [ 0  0  0 17  0  1  2 

## 7. Save your best checkpoint

Colab sessions are temporary — mount Google Drive so your trained weights survive
after the session ends or times out.

In [20]:
from google.colab import drive
drive.mount('/content/drive')
shutil.copy(BEST_PT, "/content/drive/MyDrive/skin_lesion_v4_best.pt")
print("Saved final model: 87.5% accuracy")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved final model: 87.5% accuracy


In [21]:
from google.colab import files
files.download(BEST_PT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>